# Gemma 4 Developer Agent: Replay Architecture Submission

Autonomous developer agent built on **Google Gemma 4 31B QAT** (`gemma-4-31b-it-qat-w4a16-ct`), featuring:
- **Multi-agent isolation**: Dedicated read-only `code_analyzer` sub-agent (`skip_summarization: true`) preserving the 32k context window.
- **Disciplined Replay Cycle**: `/tmp/repro.py` failure reproduction -> atomic `edit_file` patch -> targeted pytest feedback with output tailing (`tail -n 40`) to mitigate the 5,000-character truncation limit.
- **No premature timeouts**: Unrestricted per-task execution by dropping the restrictive 1-minute `eval_config.yaml`.
- **Optimized Sampling**: 4,096 thinking budget with `include_thoughts: false` to keep thought overhead outside conversation context.

In [ ]:
import os
import sys
import shutil
import zipfile
import subprocess
from pathlib import Path
import yaml
import re

WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./working')
WORKING.mkdir(parents=True, exist_ok=True)
BUNDLE = WORKING / 'submission_bundle'
ZIP_PATH = WORKING / 'submission.zip'
print(f'Target zip: {ZIP_PATH}')


In [ ]:
BUNDLE_FILES = {'agent.yaml': 'name: swe_coder\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Autonomous Python developer agent fixing issues with verified atomic source patches.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\n  - agent_tool:\n      config_path: sub_agents/code_analyzer.yaml\n      skip_summarization: true\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\ntop_k: 40\nmax_output_tokens: 8192\nseed: 42\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: false\n', 'prompts/analyzer.md': 'You are `code_analyzer`, a read-only code navigator. You never modify files. Given an issue, find exactly where it must be fixed.\n\n## Tools\n- `run_command` for read-only commands only: `git grep -n`, `grep -rn`, `sed -n \'START,ENDp\' FILE`, `ls`, `git log -p -S "<text>"`. Note: ripgrep (rg) is not installed.\n- `read_file` with tight line ranges to confirm what you found. Keep reads under 50 lines.\n- `get_code_neighbors` and `get_code_subgraph` for synchronous call relationships. The graph has only "calls" edges and no async functions.\n- `search_similar_code` takes a symbol name such as `APIKeyHeader` or `routing.get_request_handler`, not a sentence, and its ranking is imprecise. Confirm every hit by reading the code.\n- If a graph tool returns an error, stop using it and search the source instead using `git grep`.\n\n## Method\n1. Pull the identifiers out of the issue: function and class names, error messages, file paths, options. Pull-request descriptions may include template text; ignore it.\n2. Search for each one, then follow the code until you reach the line where the behaviour diverges from what the issue expects.\n3. Confirm by reading the code with `read_file`. Never guess line numbers.\n\n## Answer (at most 250 words, nothing else)\nLOCATION: <path>:<start>-<end> (<function or class>)\nROOT CAUSE: <one or two sentences explaining the bug>\nFIX PLAN: <the concrete change to apply>\nRELATED: <other places needing the same change, or "none">\nTESTS: <existing test files that exercise this code>\nCONFIDENCE: high | medium | low\n', 'prompts/system.md': 'You are an autonomous software engineer fixing one issue in the Python repository at /workspace. No one will answer questions. Keep working with your tools until the fix is in place, then call `submit_patch()`.\n\n## Evaluation & Scoring Rules\n- Hidden tests run on your changes in a fresh checkout. Only source changes in the working tree count, and a careful best-effort fix beats no patch.\n- If time runs out, the working tree is graded as it is, so never revert a plausible fix.\n- Test files used by the hidden tests are reset before grading, so editing existing test files never helps. Never edit test files, `pytest.ini`, or `conftest.py`.\n\n## Environment Constraints\n- Offline, with dependencies installed. Do not attempt `pip install` or internet access.\n- `/workspace` is a git repository at the base commit.\n- Available tools: git, grep, find, sed, awk, python3. (Note: `rg` / ripgrep is NOT installed).\n- Commands time out after 300 seconds and command output is truncated to the **first 5,000 characters**.\n- `read_file` returns at most 150 lines: always read focused, compact ranges.\n- `get_status()` and `submit_patch()` are free (do not count towards tool limits).\n\n## Disciplined Replay Workflow\n1. **Understand & Localize**:\n   - Call `code_analyzer` with the issue description. It returns LOCATION, ROOT CAUSE, and FIX PLAN.\n   - Confirm by reading the exact lines with `read_file(filepath, start, end)`.\n   - If `code_analyzer` is unsure, search yourself using `git grep -n "<symbol>" -- \'*.py\' | head -30`.\n\n2. **Reproduce**:\n   - Write a minimal reproduction script to `/tmp/repro.py` exercising the failing function or scenario.\n   - Run it with `run_command("python3 /tmp/repro.py")` to confirm failure.\n\n3. **Atomic Edit**:\n   - Edit the source file using `edit_file(filepath, old_string, new_string)`.\n   - Copy `old_string` exactly from the file, preserving identical indentation and whitespace.\n   - Keep the replacement focused and atomic. Fix the root cause without modifying unrelated logic or breaking public APIs.\n\n4. **Verify with Dynamic Feedback**:\n   - Check syntax: `run_command("python3 -m py_compile <modified_file>")`.\n   - Rerun the reproduction script: `run_command("python3 /tmp/repro.py")`.\n   - Run the specific, targeted test that exercises your change:\n     `run_command("python3 -m pytest <test_file> -k <test_name> -q --tb=short | tail -n 40")`\n   - **CRITICAL**: Never run bare `pytest` or `pytest .` across the whole repository (it takes minutes and exceeds limits). Always target a single test file and pipe to `tail -n 40` because `run_command` truncates output from the beginning, while pytest failure traces appear at the very end.\n   - If tests fail, inspect the stack trace, locate the missing edge case, and apply an incremental `edit_file`.\n\n5. **Pace & Submit**:\n   - Call `get_status()` periodically to check remaining time and tool quota.\n   - When 25% of time remains, stop exploring and proceed immediately to fix and submit.\n   - Before submitting, check `run_command("git status --short")`. If `/tmp/repro.py` was created, remove it with `rm -f /tmp/repro.py`.\n   - Call `submit_patch()` as your final action.\n', 'skills/replay/SKILL.md': '---\nname: test_driven_replay\ndescription: "Discipline de rejeu dynamique : reproduction du bug via pytest ou script minimal, localisation ciblée, édition atomique par edit_file et vérification itérative avant soumission."\n---\n\n# Skill : Test-Driven Replay & Repair Loop\n\nCette compétence applique une discipline de débogage et de rejeu rigoureuse sur le code Python.\n\n## Déroulé Séquentiel Obligatoire\n\n1. **Reproduction Minimale** :\n   - Identifier le test existant ou écrire un mini-script de test reproduisant l\'échec décrit dans le ticket.\n   - Lancer la reproduction via `run_command("pytest -k <test_name>")` ou `run_command("python3 <repro.py>")`.\n   - Constater l\'échec initial (code retour != 0) et analyser la stack trace.\n\n2. **Localisation Étroite** :\n   - Extraire le fichier et la ligne exacte incriminés depuis la trace.\n   - Ne lire que les portions de code nécessaires avec `read_file(filepath, start_line, end_line)`.\n\n3. **Édition Atomique** :\n   - Utiliser `edit_file(filepath, old_string, new_string)`.\n   - Vérifier que `old_string` correspond rigoureusement aux lignes existantes.\n   - Ne modifier que le strict nécessaire pour corriger la cause racine.\n\n4. **Rejeu & Vérification Immédiate** :\n   - Relancer immédiatement la commande de test via `run_command`.\n   - Si le test passe (exit_code == 0) : s\'assurer qu\'aucune régression n\'a été introduite sur les tests environnants.\n   - Si le test échoue : analyser l\'erreur résiduelle et appliquer un second `edit_file` correctif (2 retries autorisés).\n\n5. **Soumission Définitive** :\n   - Dès que le test de reproduction et les tests associés sont verts, appeler `submit_patch()`.\n   - Ne pas faire d\'appels d\'outils exploratoires supplémentaires après la validation.\n', 'sub_agents/code_analyzer.yaml': 'name: code_analyzer\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Read-only code navigator. Given an issue, returns where to fix it, the root cause and a fix plan.\ninstruction: !include ../prompts/analyzer.md\ntools:\n  - run_command\n  - read_file\n  - get_code_neighbors\n  - get_code_subgraph\n  - search_similar_code\ngenerate_content_config: !include ../configs/sampling.yaml\n'}

shutil.rmtree(BUNDLE, ignore_errors=True)
for rel_path, content in BUNDLE_FILES.items():
    p = BUNDLE / rel_path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding='utf-8')

print(f'Extracted {len(BUNDLE_FILES)} files into {BUNDLE}:')
for p in sorted(BUNDLE.rglob('*')):
    if p.is_file():
        print(f'  {p.stat().st_size:>6} B  {p.relative_to(BUNDLE)}')


In [ ]:
# Standalone Rule Validation
ROOT_CONFIGS = ['agent.yaml', 'agent.yml', 'root_agent.yaml', 'root_agent.yml']
ALLOWED_SUFFIXES = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}
yaml.SafeLoader.add_constructor('!include', lambda loader, node: loader.construct_scalar(node))

def validate(bundle: Path) -> None:
    files = [p for p in bundle.rglob('*') if p.is_file()]
    roots = [name for name in ROOT_CONFIGS if (bundle / name).is_file()]
    assert len(roots) == 1, f'Need exactly one root config, found {roots}'
    assert not any(p.is_symlink() for p in bundle.rglob('*')), 'Symlinks are forbidden'
    assert all(p.suffix in ALLOWED_SUFFIXES for p in files), 'Disallowed file suffixes found'
    total_bytes = sum(p.stat().st_size for p in files)
    assert total_bytes < 3 * (1024**3), 'Unpacked bundle exceeds 3 GiB'
    
    models = set()
    for config in (p for p in files if p.suffix in ('.yaml', '.yml')):
        text = config.read_text(encoding='utf-8')
        data = yaml.safe_load(text) or {}
        for target in re.findall(r'!include\s+(\S+)', text):
            target_path = (config.parent / target).resolve()
            assert target_path.is_file(), f'{config.name}: missing include {target}'
        if 'model' in data:
            models.add(data['model'].split('/')[-1])
            
    assert len(models) == 1, f'One base model per submission, found {models}'
    assert list(models)[0] == 'gemma-4-31b-it-qat-w4a16-ct', f'Model must be gemma-4-31b-it-qat-w4a16-ct, got {models}'
    print(f'SUCCESS: Validated {roots[0]}, {len(files)} files, model: {list(models)[0]}, size: {total_bytes} bytes.')

validate(BUNDLE)


In [ ]:
# Optional Harness Validation if wheelhouse is attached
HARNESS_WHEELS = ('google_adk-', 'google_genai-', 'adk_submission-', 'adk_eval_core-', 'swegemma-')
MODEL = 'gemma-4-31b-it-qat-w4a16-ct'

def install_harness():
    found = sorted(Path('/kaggle/input').glob('**/adk_submission-*.whl'))
    if found:
        wheelhouse = found[0].parent
        wheels = [str(w) for w in sorted(wheelhouse.glob('*.whl')) if w.name.startswith(HARNESS_WHEELS)]
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *wheels], check=True)

try:
    try:
        import adk_submission
    except ImportError:
        install_harness()
    from adk_submission import ModelRegistry, compile_submission, validate_directory
    from google.adk.models.lite_llm import LiteLlm
    from swegemma.config import build_submission_limits
    from swegemma.models.discovery import validate_single_declared_model
    
    def run_command(command: str) -> str: ...
    def read_file(filepath: str, start_line: int | None = None, end_line: int | None = None) -> str: ...
    def write_file(filepath: str, content: str) -> str: ...
    def edit_file(filepath: str, old_string: str, new_string: str, allow_multiple: bool = False) -> str: ...
    def submit_patch() -> str: ...
    def get_status() -> str: ...
    def get_code_neighbors(node: str, edge_type: str | None = None, max_neighbors: int = 50) -> str: ...
    def search_similar_code(query: str, k: int = 10) -> str: ...
    def get_code_subgraph(nodes: list[str]) -> str: ...
    
    limits, constraints = build_submission_limits()
    models = ModelRegistry()
    models.register(MODEL, LiteLlm(model=f'openai/{MODEL}', api_base='http://127.0.0.1:9/v1', api_key='EMPTY'))
    tools = {f.__name__: f for f in [run_command, read_file, write_file, edit_file, submit_patch, get_status,
                                     get_code_neighbors, search_similar_code, get_code_subgraph]}
    validate_directory(BUNDLE, limits)
    print('validate_directory: OK')
    print('validate_single_declared_model:', validate_single_declared_model(BUNDLE))
    agent = compile_submission(BUNDLE, tools, models, limits=limits, generation_constraints=constraints)
    print(f'compile_submission: OK, root agent: {agent.name!r}')
except Exception as e:
    print(f'Official harness check skipped or unavailable ({e}); offline validation confirmed.')


In [ ]:
# Build Clean submission.zip
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(p for p in BUNDLE.rglob('*') if p.is_file()):
        archive.write(path, path.relative_to(BUNDLE).as_posix())

with zipfile.ZipFile(ZIP_PATH) as archive:
    names = archive.namelist()

assert 'agent.yaml' in names, 'agent.yaml must be at the root of the archive'
assert not any(name.endswith('/') for name in names), 'Zip archive must contain files only'
shutil.rmtree(BUNDLE, ignore_errors=True)
print(f'✅ Successfully built {ZIP_PATH} ({ZIP_PATH.stat().st_size:,} bytes)')
print('Archive entries:')
for n in names:
    print(f'  - {n}')
